# 09 - Random forest, XGBoost y LightGBM
En este notebook evaluamos los tres modelos basados en árboles (`random_forest`, `xgboost` y `lightgbm`), con dos versiones de XGBoost con los datos reales (`data/processed/streets_unified.csv`), en las mismas particiones que el notebook 08 y con las mismas métricas (F1 macro, exactitud balanceada y kappa ponderada cuadrática).

Usamos `run_experiments` (`urban_accessibility_ml/modeling/experiments.py`), que llama a `run_one` (`train.py`) para cada partición y modelo: cada llamada entrena un modelo en una partición y devuelve una fila con las métricas.

**Importante:** `random_forest` y `lightgbm` compensan la clase A, que es muy poco frecuente, con `class_weight="balanced"`. `xgboost` no tiene ese parámetro, así que lo probamos de dos formas:
- `xgboost_balanced`: `run_one` lo entrena con pesos por tramo (`sample_weight`, calculado con `compute_sample_weight("balanced", y_train)`), que equivale a lo mismo.
- `xgboost_unbalanced`: el mismo modelo, entrenado sin pesos, para ver qué cambia el balanceo.

En la primera parte comparamos los tres modelos con `xgboost_balanced`. `xgboost_unbalanced` solo aparece en la comparación final de todos los modelos, con el mismo color que `xgboost_balanced` y con rayas.

## Carga de datos

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from urban_accessibility_ml.config import METRICS, PROCESSED_DATA_DIR, REPORTS_DIR
from urban_accessibility_ml.modeling.evaluate import summarize_folds
from urban_accessibility_ml.modeling.experiments import run_experiments, save_results
from urban_accessibility_ml.plotting.palettes import SERIE_BASE
from urban_accessibility_ml.plotting.plots import SERIES, save_figure, use_style
from urban_accessibility_ml.preprocessing.split import build_all_splits

MODELS = ["random_forest", "xgboost_balanced", "lightgbm"]
FEATURE_SET = "completo"
PROFILE = "global"
RESULTS_FILE = REPORTS_DIR / "results" / "0.09-tree-models.csv"
BASELINES = ["dummy", "logistic_regression"]
# Only in the final comparison: the base models and the XGBoost without class balancing
EXTRA_MODELS = BASELINES + ["xgboost_unbalanced"]
ALL_MODELS = ["dummy", "logistic_regression", "random_forest", "xgboost_balanced", "xgboost_unbalanced", "lightgbm"]
# One colour per model, the same as in notebook 08
MODEL_COLORS = {
    "dummy": SERIE_BASE,
    "logistic_regression": SERIES[3],
    "random_forest": SERIES[0],
    "xgboost_balanced": SERIES[1],
    "xgboost_unbalanced": SERIES[1],  # the same colour as xgboost_balanced, with stripes (MODEL_HATCH)
    "lightgbm": SERIES[2],
}
MODEL_HATCH = {"xgboost_unbalanced": "//"}
METRIC_TITLES = {
    "f1_macro": "F1 macro",
    "balanced_accuracy": "Exactitud balanceada",
    "quadratic_weighted_kappa": "Kappa ponderada cuadrática",
}

use_style()

df = pd.read_csv(PROCESSED_DATA_DIR / "streets_unified.csv")
df.shape

## Particiones
Utilizamos las mismas que en el notebook 08 (`build_all_splits`): dentro de cada ciudad (5 pliegues), entre ciudades (en los dos sentidos) y mixta (5 pliegues).

In [ ]:
splits = build_all_splits(df)
pd.Series([(s.strategy, s.name) for s in splits]).value_counts().sort_index()

## Entrenamiento y evaluación
Vamos a utilizar el método `run_experiments` de experiments.py para cada partición y modelo. Cada fila es un modelo evaluado en un único pliegue.

In [ ]:
results = run_experiments(df, splits, [FEATURE_SET], [PROFILE], MODELS)
results.round(3)

## Medias y desviaciones típicas de los pliegues
Calculamos la media y desviación típica entre pliegos para cada partición y modelo.

In [ ]:
summary = summarize_folds(results, ["strategy", "name", "model"])
summary.round(3)

## Gráficos por métrica
Vamos a mostrar la información en gráficos, como hicimos en el notebook 0.08.

In [ ]:
groups = summary[["strategy", "name"]].drop_duplicates()
labels = (groups["strategy"] + "\n" + groups["name"]).tolist()
ypos = np.arange(len(groups))
height = 0.26

fig, axes = plt.subplots(1, len(METRICS), figsize=(13, 4.6), sharey=True)
for ax, metric in zip(axes, METRICS):
    for i, model_name in enumerate(MODELS):
        s = groups.merge(summary[summary["model"] == model_name], on=["strategy", "name"])
        ax.barh(
            ypos + (i - (len(MODELS) - 1) / 2) * height, s[f"{metric}_mean"], height=height,
            xerr=s[f"{metric}_std"].fillna(0), color=MODEL_COLORS[model_name], label=model_name,
            hatch=MODEL_HATCH.get(model_name), edgecolor="white", linewidth=0,
            error_kw={"linewidth": 1},
        )
    ax.set_title(METRIC_TITLES[metric])
    ax.set_xlim(0, 1)
axes[0].set_yticks(ypos, labels)
axes[0].invert_yaxis()
handles, names = axes[0].get_legend_handles_labels()
fig.legend(handles, names, loc="lower center", ncol=len(MODELS), frameon=False)
fig.tight_layout(rect=(0, 0.06, 1, 1))
save_figure("metricas_modelos_arboles", "0.09-modelos-arboles", fig=fig)

## Comparación de todos los modelos
Añadimos los dos modelos base del notebook 08 (`dummy` y `logistic_regression`) a los tres de árboles, y añadimos también `xgboost_unbalanced` (sin pesos) para compararlo con `xgboost_balanced` (con pesos) para ver si los modelos de árboles mejoran de verdad a la referencia. Se ha asignado un color característico a cada modelo, para que se pueda ver de un vistazo qué modelo es.

In [ ]:
extra_results = run_experiments(df, splits, [FEATURE_SET], [PROFILE], EXTRA_MODELS)
all_results = pd.concat([extra_results, results], ignore_index=True)
save_results(all_results, RESULTS_FILE)
all_summary = summarize_folds(all_results, ["strategy", "name", "model"])

groups = all_summary[["strategy", "name"]].drop_duplicates()
labels = (groups["strategy"] + "\n" + groups["name"]).tolist()
ypos = np.arange(len(groups))
height = 0.13

fig, axes = plt.subplots(1, len(METRICS), figsize=(13, 6.6), sharey=True)
for ax, metric in zip(axes, METRICS):
    for i, model_name in enumerate(ALL_MODELS):
        s = groups.merge(all_summary[all_summary["model"] == model_name], on=["strategy", "name"])
        ax.barh(
            ypos + (i - (len(ALL_MODELS) - 1) / 2) * height, s[f"{metric}_mean"], height=height,
            xerr=s[f"{metric}_std"].fillna(0), color=MODEL_COLORS[model_name], label=model_name,
            hatch=MODEL_HATCH.get(model_name), edgecolor="white", linewidth=0,
            error_kw={"linewidth": 1},
        )
    ax.set_title(METRIC_TITLES[metric])
    ax.set_xlim(0, 1)
axes[0].set_yticks(ypos, labels)
axes[0].invert_yaxis()
handles, names = axes[0].get_legend_handles_labels()
fig.legend(handles, names, loc="lower center", ncol=len(ALL_MODELS), frameon=False)
fig.tight_layout(rect=(0, 0.05, 1, 1))
save_figure("metricas_cinco_modelos", "0.09-modelos-arboles", fig=fig)

Podemos comprobar que la mayoría de los modelos de árboles mejoran tanto al modelo dummy, como a la regresión logística, tanto en F1 macro, como en exactitud balanceada o kappa ponderada cuadrática.

Como excepción, en la transferencia entre ciudades, la regresión logística supera a alguno de los modelos de árboles. Por ejemplo, en la transferencia de Algeciras a Albacete, el XGBoost supera mínimamente al modelo dummy, y la regresión logística supera pŕacticamente a todos los modelos de árboles en el caso de transferencia de Albacete a Algeciras.

Aunque en la memoria habíamos indicado que íbmaos a utilizar el XGBoost balanceando las clases, parece que en este entrenamiento el desbalanceado funciona mejor, así que lo tendremos en cuenta en los posteriores análisis.